In [1]:
from pathlib import Path
from datetime import datetime
import os
import pandas as pd
import baostock as bs

# =========================
# 1. 基本参数
# =========================
START_DATE = "2020-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")
PROJECT_ROOT = Path.cwd() / "dshw-p01"

# 10只股票：覆盖5个行业，每个行业2只
STOCKS = [
    {"bs_code": "sh.600036", "code": "600036", "name": "招商银行", "industry": "银行",
     "reason": "零售银行代表性强，市场关注度高，适合作为银行板块样本。"},
    {"bs_code": "sh.601398", "code": "601398", "name": "工商银行", "industry": "银行",
     "reason": "大型国有银行代表，规模大、交易活跃，便于进行横向比较。"},

    {"bs_code": "sz.000625", "code": "000625", "name": "长安汽车", "industry": "汽车",
     "reason": "自主品牌车企代表，受行业景气与新能源转型影响明显。"},
    {"bs_code": "sh.600104", "code": "600104", "name": "上汽集团", "industry": "汽车",
     "reason": "传统整车龙头，适合与长安汽车做行业内比较。"},

    {"bs_code": "sh.600519", "code": "600519", "name": "贵州茅台", "industry": "白酒",
     "reason": "高端白酒龙头，估值和业绩都具有代表性。"},
    {"bs_code": "sz.000858", "code": "000858", "name": "五粮液", "industry": "白酒",
     "reason": "白酒核心龙头之一，可与茅台构成行业对照样本。"},

    {"bs_code": "sh.600028", "code": "600028", "name": "中国石化", "industry": "能源",
     "reason": "传统能源央企代表，受国际油价和宏观周期影响较强。"},
    {"bs_code": "sh.601857", "code": "601857", "name": "中国石油", "industry": "能源",
     "reason": "能源板块核心公司，行业代表性强，便于研究板块联动。"},

    {"bs_code": "sh.600050", "code": "600050", "name": "中国联通", "industry": "通讯",
     "reason": "基础电信运营商代表，适合观察稳健型行业特征。"},
    {"bs_code": "sz.000063", "code": "000063", "name": "中兴通讯", "industry": "通讯",
     "reason": "通信设备龙头之一，兼具成长性与行业波动特征。"},
]

# =========================
# 2. 创建项目目录
# =========================
def create_project_structure():
    folders = [
        PROJECT_ROOT / "data" / "stock",
        PROJECT_ROOT / "data" / "index",
        PROJECT_ROOT / "data" / "macro",
        PROJECT_ROOT / "data" / "finance",
        PROJECT_ROOT / "data" / "clean",
        PROJECT_ROOT / "data" / "combined",
        PROJECT_ROOT / "output",
    ]
    for folder in folders:
        os.makedirs(folder, exist_ok=True)

    # 创建日志文件和README占位文件
    (PROJECT_ROOT / "download_log.txt").touch(exist_ok=True)
    if not (PROJECT_ROOT / "README.md").exists():
        (PROJECT_ROOT / "README.md").write_text("# dshw-p01\n", encoding="utf-8")


# =========================
# 3. 日志函数
# =========================
def write_log(status: str, task_name: str, message: str):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = f"[{now}] {status:<8} {task_name:<15} {message}\n"
    with open(PROJECT_ROOT / "download_log.txt", "a", encoding="utf-8") as f:
        f.write(log_line)


# =========================
# 4. 导出股票清单
# =========================
def save_stock_universe():
    df = pd.DataFrame(STOCKS)
    df = df[["code", "bs_code", "name", "industry", "reason"]]
    df.to_csv(PROJECT_ROOT / "data" / "stock" / "stock_universe.csv",
              index=False, encoding="utf-8-sig")


def save_readme_snippet():
    lines = []
    lines.append("## 1.1 自选股票清单")
    lines.append("")
    lines.append("| 股票代码 | 股票名称 | 行业 | 选股理由 |")
    lines.append("|---|---|---|---|")
    for s in STOCKS:
        lines.append(f"| {s['code']} | {s['name']} | {s['industry']} | {s['reason']} |")
    lines.append("")
    lines.append("## 数据来源说明")
    lines.append("")
    lines.append("- 本部分股票日度行情数据使用 `baostock` 获取。")
    lines.append("- 接口：`bs.query_history_k_data_plus()`")
    lines.append("- 复权方式：后复权（`adjustflag='1'`）")
    content = "\n".join(lines)

    (PROJECT_ROOT / "stock_selection_for_readme.md").write_text(
        content, encoding="utf-8"
    )


# =========================
# 5. 下载单只股票
# =========================
def download_one_stock(stock: dict):
    task_name = f"stock_{stock['code']}"

    fields = "date,code,open,high,low,close,volume,amount,tradestatus"
    rs = bs.query_history_k_data_plus(
        code=stock["bs_code"],
        fields=fields,
        start_date=START_DATE,
        end_date=END_DATE,
        frequency="d",
        adjustflag="1"   # 1 = 后复权
    )

    if rs.error_code != "0":
        raise RuntimeError(f"baostock返回错误：{rs.error_msg}")

    rows = []
    while (rs.error_code == "0") and rs.next():
        rows.append(rs.get_row_data())

    if len(rows) == 0:
        raise ValueError("No data returned")

    df = pd.DataFrame(rows, columns=rs.fields)

    # 过滤停牌日，只保留正常交易
    df = df[df["tradestatus"] == "1"].copy()
    if df.empty:
        raise ValueError("No valid trading rows after filtering tradestatus")

    # 数据类型处理
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df["code"] = stock["code"]

    numeric_cols = ["open", "high", "low", "close", "volume", "amount"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 调整列顺序
    df = df[["date", "code", "open", "close", "high", "low", "volume", "amount"]]
    df = df.sort_values("date").reset_index(drop=True)

    save_path = PROJECT_ROOT / "data" / "stock" / f"stock_{stock['code']}.csv"
    df.to_csv(save_path, index=False, encoding="utf-8-sig")

    write_log("SUCCESS", task_name, f"shape={df.shape}")
    return df


# =========================
# 6. 主程序
# =========================
def main():
    print("开始创建项目目录...")
    create_project_structure()

    print("保存股票清单...")
    save_stock_universe()
    save_readme_snippet()

    print("登录 baostock...")
    lg = bs.login()
    if lg.error_code != "0":
        raise RuntimeError(f"登录失败：{lg.error_code}, {lg.error_msg}")

    success_count = 0
    fail_count = 0

    try:
        for stock in STOCKS:
            try:
                df = download_one_stock(stock)
                success_count += 1
                print(f"下载成功：{stock['code']} {stock['name']}，共 {len(df)} 行")
            except Exception as e:
                fail_count += 1
                task_name = f"stock_{stock['code']}"
                write_log("FAILED", task_name, f"Error: {str(e)}")
                print(f"下载失败：{stock['code']} {stock['name']} -> {e}")
    finally:
        bs.logout()

    print("\n全部完成")
    print(f"成功：{success_count} 只")
    print(f"失败：{fail_count} 只")
    print(f"项目路径：{PROJECT_ROOT}")


if __name__ == "__main__":
    main()

开始创建项目目录...
保存股票清单...
登录 baostock...
login success!
下载成功：600036 招商银行，共 1514 行
下载成功：601398 工商银行，共 1514 行
下载成功：000625 长安汽车，共 1514 行
下载成功：600104 上汽集团，共 1514 行
下载成功：600519 贵州茅台，共 1514 行
下载成功：000858 五粮液，共 1514 行
下载成功：600028 中国石化，共 1514 行
下载成功：601857 中国石油，共 1514 行
下载成功：600050 中国联通，共 1514 行
下载成功：000063 中兴通讯，共 1513 行
logout success!

全部完成
成功：10 只
失败：0 只
项目路径：d:\Microsoft VS Code\dshw-p01
